# 09 — Design Space: δ_M vs x/d (mean material values)

This notebook reproduces the manuscript figure `ch10_delta_vs_xd` —
the redistribution factor $\delta_M$ as a function of the hogging neutral-axis
depth ratio $x/d$ — using **mean material values** per EC2 §5.7(NA.10).

**Section layout:** symmetric doubly-reinforced rectangular section
($A_{s,\text{top}} = A_{s,\text{bot}} = A_s$) with $b \times h = 300 \times 600\,\text{mm}$,
cover $c = 50\,\text{mm}$, left span $L_a = 6\,000\,\text{mm}$.

**Mean material conventions (EC2 §5.7(NA.10)):**

| Property | Notation | Value | Source |
|----------|----------|-------|---------|
| Concrete | $f_{cm}$ | 38 MPa | Sargin parabola, $f_{cm} = f_{ck} + 8$ |
| Steel yield | $f_{yR}$ | 550 MPa | $f_{yR} = 1.1\,f_{yk}$, NA.10 |
| Steel tensile | $f_{tR}$ | 594 MPa | $f_{tR} = 1.08\,f_{yR}$, B500B |
| Partial factors | $\gamma_c, \gamma_s$ | 1.0 | mean prediction |

**x-axis convention:** the neutral-axis depth ratio on the x-axis uses the EC2
singly-reinforced formula with design values —
$x/d = \omega/0.8$, $\omega = A_s/(b\,d) \cdot f_{yd}/f_{cd}$ — identical to the
manuscript figure, so that mean and design curves can be compared directly.
For the symmetric doubly-reinforced section, the *actual* M-κ neutral-axis depth
is much shallower (compression steel partially cancels tension steel; see the
second panel).

**Three span ratios:** $S = L_b/L_a \in \{0.7, 1.0, 1.3\}$.

**Filter:** only hogging-governed configurations
(`cba.failure_mode_u == 'hog'`) are plotted.

**EC2 limit (eq. 5.10a, German NA):**
$$\delta_{\min} = 0.64 + 0.8\,\frac{x_u}{d}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis
from scite.beam.continuous import ContinuousBeamAnalysis, CompatibilityNoRoot
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.shapes import RectangularShape
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_concrete import EC2Concrete
from scite.matmod.steel_reinforcement import SteelReinforcement
from scite.mkappa.mkappa import MKappaAnalysis

## Parameters

Material constants are defined for both modes so that the EC2 design formula
x/d can be computed on the same x-axis as the manuscript figure.

In [ ]:
# ── Design values (for EC2 x/d formula on x-axis only) ───────────────────────
f_ck     = 30.0
alpha_cc = 0.85;  gamma_c = 1.5
f_cd     = alpha_cc * f_ck / gamma_c         # 17.0 N/mm²
f_yk     = 500.0; gamma_s = 1.15
f_yd     = f_yk / gamma_s                    # 434.8 N/mm²

# ── Mean values — EC2 §5.7(NA.10) B500B ──────────────────────────────────────
f_cm  = f_ck + 8.0                           # 38 N/mm²
f_yR  = 1.1  * f_yk                          # 550 N/mm²
f_tR  = 1.08 * f_yR                          # 594 N/mm²
eps_ud = 0.05

concrete_m = EC2Concrete(f_ck=f_ck, factor=1.0)
steel_m    = SteelReinforcement(f_yk=f_yR, f_tk=f_tR, gamma_s=1.0,
                                E_s=200_000.0, eps_ud=eps_ud)

# ── Design materials for M-κ model ───────────────────────────────────────────
# EC2Concrete.factor scales the Sargin peak stress (factor × f_cm).
# To reproduce f_cd = alpha_cc × f_ck / gamma_c = 17 MPa as the peak stress:
#   factor = f_cd / f_cm = 17 / 38
concrete_d = EC2Concrete(f_ck=f_ck, factor=(alpha_cc * f_ck / gamma_c) / (f_ck + 8.0))
steel_d    = SteelReinforcement(f_yk=f_yk, f_tk=1.08 * f_yk, gamma_s=gamma_s,
                                E_s=200_000.0, eps_ud=0.025)

# ── Section geometry ─────────────────────────────────────────────────────────
b  = 300.0;  h = 600.0;  c = 50.0;  d = h - c   # d = 550 mm
La = 6_000.0

print(f"Design: f_cd = {f_cd:.1f} N/mm²,  f_yd = {f_yd:.1f} N/mm²")
print(f"Mean:   f_cm = {f_cm:.0f} N/mm²,  f_yR = {f_yR:.0f} N/mm²,  f_tR = {f_tR:.0f} N/mm²")
print(f"Design M-κ:  peak concrete stress = {concrete_d.factor * concrete_d.f_cm:.1f} N/mm²  "
      f"(target f_cd = {f_cd:.1f} N/mm²)")

## Analysis helpers

The section is **symmetric** ($A_{s,\text{top}} = A_{s,\text{bot}} = A_s$).
This matches the approach in notebook 08 (design mode) and the manuscript figure.

In [ ]:
def _build_cs(A_s: float) -> CrossSection:
    """Singly-reinforced section with mean materials (top bar only)."""
    return CrossSection(
        shape=RectangularShape(b=b, h=h),
        concrete=concrete_m,
        reinforcement=ReinforcementLayout(layers=[
            ReinforcementLayer(z=h - c, A_s=A_s, material=steel_m),
        ]),
    )


def _build_cs_design(A_s: float) -> CrossSection:
    """Singly-reinforced section with design materials (top bar only)."""
    return CrossSection(
        shape=RectangularShape(b=b, h=h),
        concrete=concrete_d,
        reinforcement=ReinforcementLayout(layers=[
            ReinforcementLayer(z=h - c, A_s=A_s, material=steel_d),
        ]),
    )


def make_cba(A_s: float, S: float = 1.0) -> ContinuousBeamAnalysis:
    cs = _build_cs(A_s)
    bda_left = BeamDeflectionAnalysis(
        cs=cs, L=La, load_type='dist', n_kappa=150, solve_mode='both',
    )
    if S == 1.0:
        bda_right = bda_left
    else:
        bda_right = BeamDeflectionAnalysis(
            cs=cs, L=S * La, load_type='dist', n_kappa=150, solve_mode='both',
        )
    return ContinuousBeamAnalysis(span_left=bda_left, span_right=bda_right, q=1.0)


def xd_ec2_design(A_s: float) -> float:
    """EC2 singly-reinforced x/d from design values — used as x-axis (same as manuscript)."""
    omega = (A_s / (b * d)) * (f_yd / f_cd)
    return omega / 0.8

## Verification — single point

Quick smoke test: $A_s = 1000\,\text{mm}^2$, $S = 1$.

In [ ]:
A_s_check = 1000.0
cba_check  = make_cba(A_s_check, S=1.0)

xd_f  = xd_ec2_design(A_s_check)
xd_mk = cba_check.xd_hog_u
dM    = cba_check.delta_M_u
mode  = cba_check.failure_mode_u

print(f"A_s = {A_s_check:.0f} mm²,  S = 1.0")
print(f"  x/d  EC2 design formula  = {xd_f:.4f}")
print(f"  x/d  M-κ model (actual)  = {xd_mk:.4f}  (symmetric section — much shallower)")
print(f"  δ_M                      = {dM:.4f}")
print(f"  failure_mode_u           = '{mode}'")
print(f"  EC2 δ_min (design x/d)   = {0.64 + 0.8*xd_f:.4f}")

## Step 1: x/d comparison — M-κ model vs EC2 formula

For the symmetric doubly-reinforced section, the actual M-κ neutral-axis depth
in hogging is much shallower than the EC2 singly-reinforced formula, because the
compression steel (bottom bars, in compression at the interior support) partially
cancels the tension steel force.  This gap is even larger for mean materials
(stronger concrete → shallower compression zone) than for design materials.

With mean materials:
$$\frac{x}{d}\bigg|_{\text{mean, SR}} = \frac{A_s \, f_{yR}}{0.8\,b\,d\,f_{cm}}$$

compared to the design formula on the x-axis:
$$\frac{x}{d}\bigg|_{\text{design, SR}} = \frac{A_s \, f_{yd}}{0.8\,b\,d\,f_{cd}}$$

In [ ]:
# A_s range: EC2 design formula x/d = 0.10 → 0.50 (same as manuscript)
A_s_range = np.linspace(200, 2600, 13)

xd_design_arr    = []
xd_mean_sr_arr   = []
xd_mk_arr        = []
xd_mk_design_arr = []

for A_s in A_s_range:
    xd_d  = xd_ec2_design(A_s)
    xd_ms = (A_s / (b * d)) * (f_yR / f_cm) / 0.8   # mean, singly-reinforced formula

    mk_m = MKappaAnalysis(cs=_build_cs(A_s), n_kappa=60, solve_mode='negative')
    mk_m.solve()
    xd_mk_mean_val = mk_m.xd_neg

    mk_d = MKappaAnalysis(cs=_build_cs_design(A_s), n_kappa=60, solve_mode='negative')
    mk_d.solve()
    xd_mk_design_val = mk_d.xd_neg

    xd_design_arr.append(xd_d)
    xd_mean_sr_arr.append(xd_ms)
    xd_mk_arr.append(xd_mk_mean_val)
    xd_mk_design_arr.append(xd_mk_design_val)
    print(f"A_s={A_s:6.0f}  xd_design={xd_d:.3f}  xd_mean_SR={xd_ms:.3f}  "
          f"xd_MK_mean={xd_mk_mean_val:.3f}  xd_MK_design={xd_mk_design_val:.3f}")

xd_design_arr    = np.array(xd_design_arr)
xd_mean_sr_arr   = np.array(xd_mean_sr_arr)
xd_mk_arr        = np.array(xd_mk_arr)
xd_mk_design_arr = np.array(xd_mk_design_arr)

In [ ]:
from pathlib import Path

FIG_OUT = Path('/home/rch/Coding/icc_manuscript/lecture_notes/fig')

fig, ax = plt.subplots(figsize=(7.5, 4.8))

# Design: formula (solid) vs M-κ model (dashed)
# Note: design M-κ underperforms for very small A_s (solver numerical issue);
# drop the A_s=200 design-MK point (mask: xd_MK_design deviates strongly from formula)
valid_d = np.abs(xd_mk_design_arr - xd_design_arr) < 0.15
ax.plot(A_s_range, xd_design_arr,              'o-',  color='#d62728', lw=1.8, ms=6,
        label=r'Design formula: $\omega/0.8$, $f_{yd}/f_{cd}$')
ax.plot(A_s_range[valid_d], xd_mk_design_arr[valid_d], 'o--', color='#d62728', lw=1.8, ms=6,
        alpha=0.55, label=r'M-$\kappa$ model, design materials')

# Mean: formula (solid) vs M-κ model (dashed)
ax.plot(A_s_range, xd_mean_sr_arr, 's-',  color='#2ca02c', lw=1.8, ms=6,
        label=r'Mean formula: $\omega/0.8$, $f_{yR}/f_{cm}$')
ax.plot(A_s_range, xd_mk_arr,      's--', color='#2ca02c', lw=1.8, ms=6,
        alpha=0.55, label=r'M-$\kappa$ model, mean materials')

ax.set_xlabel(r'$A_s$ [mm²]  (top bar, singly reinforced at hogging section)', fontsize=10)
ax.set_ylabel(r'$x/d$', fontsize=11)
ax.set_title(
    r'$A_s \to x/d$: EC2 stress-block formula vs M-$\kappa$ model  (C30/37, B500B)',
    fontsize=10,
)
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.25, ls='--')
plt.tight_layout()
fig.savefig(FIG_OUT / 'ch10_xd_as_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved  ch10_xd_as_comparison.png")

## Stress/Force Profiles at the Hogging M–κ Peak — Two Extremes

Strain and stress distributions for the **smallest** and **largest** $A_s$ values in the study,
at the peak of the negative M–κ branch (hogging section under design materials).

- **Small $A_s = 200$ mm²**: very shallow compression zone, steel reaches $\varepsilon_{ud}$ first → ductile
- **Large $A_s = 2600$ mm²**: deep compression zone, concrete crushes first → brittle

Figures exported as `ch10_xd_small.png` and `ch10_xd_large.png` for the lecture slides.

In [ ]:
from scite.cs_design.cs_stress_strain_profile import StressStrainProfile

_profile_cases = [
    (200.0,  'small', 'ch10_xd_small.png'),
    (2600.0, 'large', 'ch10_xd_large.png'),
]

for A_s, label, fname in _profile_cases:
    # Use mean materials — design-MK solver has numerical issues for very small A_s
    cs = _build_cs(A_s)
    mk = MKappaAnalysis(cs=cs, n_kappa=80, solve_mode='negative')
    mk.solve()

    idx_pk     = int(np.argmin(mk.M_neg_values))
    kappa_pk   = float(mk.kappa_neg_values[idx_pk])
    eps_bot_pk = float(mk.eps_bot_neg_values[idx_pk])
    M_pk       = float(mk.M_neg_values[idx_pk])
    xd_val     = mk.xd_neg
    # Design-formula x/d for reference label
    xd_formula = xd_ec2_design(A_s)

    profile = StressStrainProfile(cs, kappa=kappa_pk, eps_bottom=eps_bot_pk)

    fig, (ax_strain, ax_stress) = plt.subplots(
        1, 2, figsize=(10, 5), sharey=True,
        gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    )
    profile.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)

    # Increase font sizes for slide legibility
    for ax in (ax_strain, ax_stress):
        ax.tick_params(labelsize=12)
        ax.xaxis.label.set_size(13)
        ax.yaxis.label.set_size(13)
        ax.title.set_size(13)

    fig.suptitle(
        rf'$A_s = {A_s:.0f}\ \mathrm{{mm}}^2$ | $x/d = {xd_val:.3f}$ (mean M-$\kappa$) '
        rf'/ {xd_formula:.3f} (EC2 formula) | '
        rf'$M_{{pk}} = {M_pk:.1f}\ \mathrm{{kN\cdot m}}$  — {label} reinforcement',
        fontsize=10,
    )
    plt.tight_layout()
    fig.savefig(FIG_OUT / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved  {fname}  (A_s={A_s:.0f} mm², x/d_MK={xd_val:.3f}, x/d_formula={xd_formula:.3f})")

## Step 2: δ_M sweep × span ratio S (mean materials)

For each $(A_s,\, S)$ compute `cba.delta_M_u` and `cba.failure_mode_u` with mean
material models.  The x-axis value is the EC2 **design** formula x/d — same
parameterisation as the manuscript figure — allowing direct comparison between
the mean-mode and design-mode redistribution curves.

In [ ]:
S_values = [0.7, 1.0, 1.3]

results = {S: {'xd': [], 'delta_M': [], 'mode': []} for S in S_values}

for S in S_values:
    print(f"\n── S = {S:.1f}  (L_b = {S*La/1000:.1f} m) ───────────────────────────────")
    for A_s in A_s_range:
        xd_f = xd_ec2_design(A_s)
        try:
            cba  = make_cba(A_s, S=S)
            dM   = cba.delta_M_u
            mode = cba.failure_mode_u
        except CompatibilityNoRoot:
            dM   = np.nan
            mode = 'err'
        results[S]['xd'].append(xd_f)
        results[S]['delta_M'].append(dM)
        results[S]['mode'].append(mode)
        print(f"  A_s={A_s:6.0f} mm²  x/d_EC2={xd_f:.3f}  δ_M={dM:.4f}  mode={mode}")

    results[S]['xd']      = np.array(results[S]['xd'])
    results[S]['delta_M'] = np.array(results[S]['delta_M'])

## Figure: δ_M vs x/d (mean mode, hogging-governed)

Reproduces `ch10_delta_vs_xd` with mean material values.  Only hogging-governed
points are shown.  The EC2 design lower bound $\delta_{\min} = 0.64 + 0.8\,x/d$
is shown as a reference — note that the simulation x/d (from M-κ model) is much
smaller than the x-axis formula value for a symmetric doubly-reinforced section.

In [ ]:
COLORS  = {0.7: '#2166ac', 1.0: '#1a9641', 1.3: '#d7191c'}
MARKERS = {0.7: 'o',       1.0: 's',       1.3: '^'}

xd_ref    = np.linspace(0.05, 0.52, 300)
delta_ec2 = 0.64 + 0.8 * xd_ref

fig, ax = plt.subplots(figsize=(7.5, 5))

for S in S_values:
    mask = np.array(results[S]['mode']) == 'hog'
    xd   = results[S]['xd'][mask]
    dM   = results[S]['delta_M'][mask]
    ax.plot(xd, dM, MARKERS[S] + '-',
            color=COLORS[S], lw=1.8, ms=6,
            label=rf'$S = {S:.1f}$  (hog-governed, mean mode)')

ax.plot(xd_ref, delta_ec2, '--', color='#888888', lw=1.5,
        label=r'EC2 §5.5 eq. (5.10a): $\delta_{\min} = 0.64 + 0.8\,x_u/d$')
ax.axhline(1.0, color='gray', lw=0.8, ls=':', label=r'$\delta = 1$ (elastic)')
ax.axhline(0.7, color='gray', lw=0.8, ls=':', label=r'$\delta = 0.70$ (B500B limit)')

ax.set_xlabel(
    r'$x/d$ — EC2 design formula  ($\omega/0.8$, $f_{yd}/f_{cd}$)'
    '\n'
    r'(actual M-$\kappa$ $x/d$ for symmetric doubly-reinforced section is much shallower)',
    fontsize=9,
)
ax.set_ylabel(r'$\delta_M = |M_{\mathrm{nl,hog}}| / |M_{\mathrm{el,hog}}|$', fontsize=11)
ax.set_title(
    r'$\delta_M$ vs $x/d$ — mean material values (C30/37: $f_{cm}=38$ MPa, '
    r'B500B: $f_{yR}=550$ MPa)',
    fontsize=10,
)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0.05, 0.52)
ax.set_ylim(0.55, 1.05)
ax.grid(True, alpha=0.25, ls='--')
plt.tight_layout()
plt.show()

## Summary table (S = 1.0, hogging-governed)

In [ ]:
S_ref = 1.0
mask  = np.array(results[S_ref]['mode']) == 'hog'
xd_s1 = results[S_ref]['xd'][mask]
dM_s1 = results[S_ref]['delta_M'][mask]

print(f"{'A_s':>8}  {'x/d_EC2':>8}  {'δ_M':>7}  {'δ_EC2':>7}  {'margin':>8}")
print("-" * 52)
for A_s, xd, dM in zip(A_s_range[mask], xd_s1, dM_s1):
    ec2 = 0.64 + 0.8 * xd
    print(f"  {A_s:5.0f} mm²  {xd:.3f}     {dM:.4f}   {ec2:.4f}   {dM - ec2:+.4f}")

## Failure-mode map — which failure governs at each (A_s, S)

In [ ]:
mode_color = {'hog': '#2ca02c', 'sag_a': '#d62728', 'sag_b': '#ff7f0e', 'err': '#999999'}

fig, axes = plt.subplots(1, len(S_values), figsize=(12, 3.5), sharey=True)

for ax, S in zip(axes, S_values):
    modes  = results[S]['mode']
    xd_all = results[S]['xd']
    dM_all = results[S]['delta_M']
    for mode in ['hog', 'sag_a', 'sag_b', 'err']:
        mask = np.array(modes) == mode
        if not np.any(mask):
            continue
        ax.scatter(xd_all[mask], dM_all[mask],
                   color=mode_color[mode], s=55, label=mode, zorder=3)
    ax.plot(xd_ref, delta_ec2, '--', color='#888888', lw=1.2)
    ax.axhline(1.0, color='gray', lw=0.7, ls=':')
    ax.set_title(rf'$S = {S:.1f}$')
    ax.set_xlabel(r'$x/d$  (design formula)')
    ax.set_xlim(0.05, 0.55)
    ax.set_ylim(0.55, 1.10)
    ax.grid(True, alpha=0.2, ls='--')
    ax.legend(fontsize=8, title='failure mode')

axes[0].set_ylabel(r'$\delta_M$')
fig.suptitle('Failure-mode map — mean material values', fontsize=11)
plt.tight_layout()
plt.show()